In [ ]:
import sys
from pathlib import Path

EXTRA_SITE_PACKAGES = "/global/homes/a/atsouros/.local/perlmutter/pytorch2.11.0/lib/python3.12/site-packages"
if EXTRA_SITE_PACKAGES not in sys.path:
    sys.path.append(EXTRA_SITE_PACKAGES)

import os
import re
from collections import defaultdict

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from astropy.io import fits

print("healpy:", hp.__version__)

In [ ]:
VERSION = 4
TARGET_NSIDE = 1024
BORDERPIX = 64
KEEPBORDER = 32
SUBDIVIDE = 4
USE_WEIGHTED = False

RECOVERED_PATCHES_DIR = Path(f"/pscratch/sd/a/atsouros/STL/planck_results/version_{VERSION}")
NUISANCE_PATCHES_DIR = Path("/pscratch/sd/e/erussie/GNILC+ST/patches/nuisance")
OBSERVED_FITS_IN = Path("/pscratch/sd/e/erussie/GNILC/data/preproc/input_maps/IQU_353_map_wo_CMB_wiener_wo_dip_zodi_MJy_sr_10_arcmin.fits")
RESOURCES_DIR = Path("/pscratch/sd/a/atsouros/STL/patches2map/resources")
OUTPUT_DIR = Path("/pscratch/sd/a/atsouros/STL/patches2map/output/minimal_noise_beam_match")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FITS_PIXFACES = RESOURCES_DIR / f"pixfaces_nside={TARGET_NSIDE}_subdivide={SUBDIVIDE}_borderpix={BORDERPIX}.fits"

OBSERVED_FITS = OUTPUT_DIR / f"observed_IQU_nside{TARGET_NSIDE}.fits"
RECOVERED_FITS = OUTPUT_DIR / f"recovered_v{VERSION}_IQU_nside{TARGET_NSIDE}_weighted={USE_WEIGHTED}.fits"
RECOVERED_NOISE_BEAM_FITS = OUTPUT_DIR / f"recovered_v{VERSION}_white_noise_beam_matched_IQU_nside{TARGET_NSIDE}.fits"

NUISANCE_FREQ = 353
NUISANCE_VERSION = "v4_10_arcmin"
NUISANCE_MAP_SIZE = 384
ALL_NUISANCE_FITS = OUTPUT_DIR / (
    f"nuisance_all_Q{NUISANCE_FREQ}_U{NUISANCE_FREQ}_"
    f"{NUISANCE_VERSION}_IQU_nside{TARGET_NSIDE}_weighted={USE_WEIGHTED}.fits"
)

LMAX = 2 * TARGET_NSIDE
FIT_LMIN = 30
FIT_LMAX = 1400
FWHM_GRID_ARCMIN = np.linspace(0.0, 40.0, 81)  # change if needed
RANDOM_SEED = 1234

print("Observed FITS out :", OBSERVED_FITS)
print("Recovered FITS out:", RECOVERED_FITS)
print("Matched FITS out  :", RECOVERED_NOISE_BEAM_FITS)
print("All nuisance FITS:", ALL_NUISANCE_FITS)


In [ ]:
def list_recovered_patch_files_by_index(path, valid_indices):
    regex = re.compile(r"^p(\d+)_")
    valid_indices = set(int(i) for i in valid_indices)
    grouped = defaultdict(list)

    for fname in sorted(os.listdir(path)):
        if not fname.endswith(".npy"):
            continue
        m = regex.match(fname)
        if m is None:
            continue
        idx = int(m.group(1))
        if idx in valid_indices:
            grouped[idx].append(Path(path) / fname)

    if not grouped:
        raise FileNotFoundError(f"No recovered patch files found in {path}")

    indices = np.array(sorted(grouped.keys()), dtype=int)
    print(f"Recovered: {sum(len(v) for v in grouped.values())} files for {len(indices)} patch indices")
    return indices, grouped


def load_grouped_patches(indices, grouped):
    patches = []
    for idx in indices:
        arrs = []
        for f in grouped[int(idx)]:
            a = np.load(f)
            if a.ndim == 2:
                a = a[None, :, :]
            arrs.append(a)
        patches.append(np.asarray(arrs, dtype=float).mean(axis=0))
    patches = np.asarray(patches, dtype=float)
    if patches.ndim == 3:
        patches = patches[:, None, :, :]
    print("Recovered patches:", patches.shape)
    return patches


def cosine_weights(ny, nx, keepborder):
    wy = np.ones(ny)
    wx = np.ones(nx)
    if keepborder > 0:
        for i in range(keepborder):
            w = 0.5 * (1.0 - np.cos(np.pi * (i + 0.5) / keepborder))
            wy[i] = wy[ny - 1 - i] = w
            wx[i] = wx[nx - 1 - i] = w
    return np.outer(wy, wx)


def stitch_qu(Q_patches, U_patches, pixfaces, weights):
    npix = hp.nside2npix(TARGET_NSIDE)
    qnum = np.zeros(npix)
    unum = np.zeros(npix)
    wden = np.zeros(npix)

    idx = pixfaces.reshape(-1)
    Q = Q_patches.reshape(-1)
    U = U_patches.reshape(-1)
    W = np.tile(weights, (Q_patches.shape[0], 1, 1)).reshape(-1) if USE_WEIGHTED else np.ones_like(Q)

    good = (idx >= 0) & (idx < npix) & np.isfinite(Q) & np.isfinite(U) & np.isfinite(W) & (W > 0)
    np.add.at(qnum, idx[good], Q[good] * W[good])
    np.add.at(unum, idx[good], U[good] * W[good])
    np.add.at(wden, idx[good], W[good])

    mask = wden > 0
    out = np.full((3, npix), hp.UNSEEN, dtype=float)
    out[0, mask] = 0.0
    out[1, mask] = qnum[mask] / wden[mask]
    out[2, mask] = unum[mask] / wden[mask]
    print(f"Recovered covered pixels: {mask.sum():,}/{npix:,} ({mask.mean():.1%})")
    return out, mask


def clean_map(m):
    m = np.asarray(m, dtype=float).copy()
    m[(m == hp.UNSEEN) | ~np.isfinite(m)] = 0.0
    return m


def common_mask_from_maps(*maps):
    masks = []
    for m in maps:
        masks.append(np.all(np.isfinite(m) & (m != hp.UNSEEN), axis=0))
    return np.logical_and.reduce(masks)


def scalar_cl(map_1d, mask, lmax):
    x = np.asarray(map_1d, dtype=float).copy()
    x[(x == hp.UNSEEN) | ~np.isfinite(x)] = 0.0
    x[~mask] = 0.0
    return hp.anafast(x, lmax=lmax)


def dl_from_cl(cl):
    ell = np.arange(len(cl))
    return ell, ell * (ell + 1) * cl / (2 * np.pi)


def qq_uu_cls(map_iqu, mask, lmax):
    return scalar_cl(map_iqu[1], mask, lmax), scalar_cl(map_iqu[2], mask, lmax)


def save_iqu(path, m):
    hp.write_map(
        str(path), m, overwrite=True,
        column_names=["I", "Q", "U"],
        column_units=["MJy/sr", "MJy/sr", "MJy/sr"],
        extra_header=[("COORDSYS", "G"), ("BUNIT", "MJy/sr")],
    )
    print("Saved:", path)



def downsample_by_four(image):
    image = np.asarray(image, dtype=float)
    h, w = image.shape[-2:]
    if h % 2 != 0 or w % 2 != 0:
        raise ValueError(f"Image dimensions must be even for 2x2 downsampling, got {image.shape}.")
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D nuisance patch, got {image.shape}.")
    return image.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3))


NUISANCE_RE = re.compile(
    r"^patch_(\d+)_noise_([QU])" + str(NUISANCE_FREQ) +
    r"_.*noise_seed_(\d+)_CMB_res_seed_(\d+)_" + re.escape(NUISANCE_VERSION) + r"\.npy$"
)


def index_nuisance_files(valid_indices):
    valid_indices = set(int(i) for i in valid_indices)
    path_by_key = {}
    seeds_by_patch_stokes = defaultdict(set)

    for path in sorted(NUISANCE_PATCHES_DIR.glob(f"patch_*_noise_[QU]{NUISANCE_FREQ}_*_{NUISANCE_VERSION}.npy")):
        match = NUISANCE_RE.match(path.name)
        if match is None:
            continue
        patch_idx = int(match.group(1))
        if patch_idx not in valid_indices:
            continue
        stokes = match.group(2)
        noise_seed = int(match.group(3))
        cmb_res_seed = int(match.group(4))
        key = (patch_idx, stokes, noise_seed, cmb_res_seed)
        if key in path_by_key:
            raise RuntimeError(f"Duplicate nuisance file for {key}: {path_by_key[key]} and {path}")
        path_by_key[key] = path
        seeds_by_patch_stokes[(patch_idx, stokes)].add((noise_seed, cmb_res_seed))

    common_keys = None
    missing = []
    for patch_idx in sorted(valid_indices):
        for stokes in ("Q", "U"):
            keys = seeds_by_patch_stokes.get((patch_idx, stokes), set())
            if not keys:
                missing.append((patch_idx, stokes))
            common_keys = set(keys) if common_keys is None else common_keys & keys

    if missing:
        raise FileNotFoundError(f"Missing {NUISANCE_VERSION} nuisance files for patch/stokes entries: {missing[:10]}")
    common_keys = sorted(common_keys or [])
    if not common_keys:
        raise RuntimeError("No globally consistent nuisance seed pair exists across all patches and Q/U maps.")

    print(
        f"Indexed {len(path_by_key)} nuisance files; "
        f"found {len(common_keys)} globally consistent seed pairs."
    )
    return path_by_key, common_keys


def load_nuisance_patches_from_index(indices, path_by_key, noise_seed, cmb_res_seed):
    q_patches = []
    u_patches = []
    target_shape = (NUISANCE_MAP_SIZE, NUISANCE_MAP_SIZE)
    for idx in indices:
        q_path = path_by_key[(int(idx), "Q", int(noise_seed), int(cmb_res_seed))]
        u_path = path_by_key[(int(idx), "U", int(noise_seed), int(cmb_res_seed))]
        q = np.load(q_path).astype(float)
        u = np.load(u_path).astype(float)
        if q.shape == (2 * NUISANCE_MAP_SIZE, 2 * NUISANCE_MAP_SIZE):
            q = downsample_by_four(q)
        if u.shape == (2 * NUISANCE_MAP_SIZE, 2 * NUISANCE_MAP_SIZE):
            u = downsample_by_four(u)
        if q.shape != target_shape or u.shape != target_shape:
            raise ValueError(
                f"Patch {idx} nuisance shape mismatch after downgrade: Q={q.shape}, U={u.shape}, expected {target_shape}."
            )
        q_patches.append(q)
        u_patches.append(u)
    return np.asarray(q_patches, dtype=float), np.asarray(u_patches, dtype=float)


def nuisance_extension_name(noise_seed, cmb_res_seed):
    return f"N{int(noise_seed):04d}C{int(cmb_res_seed):02d}"


def write_all_nuisance_realizations(path, pixfaces_all):
    if path.exists():
        print("All-realization nuisance FITS already exists, skipping:", path)
        return

    indices = np.arange(pixfaces_all.shape[0])
    path_by_key, seed_pairs = index_nuisance_files(indices)
    trim = BORDERPIX - KEEPBORDER
    pixfaces = pixfaces_all[indices]
    if trim > 0:
        pixfaces = pixfaces[:, trim:-trim, trim:-trim]
    weights = cosine_weights(pixfaces.shape[1], pixfaces.shape[2], KEEPBORDER)

    primary = fits.PrimaryHDU()
    primary.header["NSIDE"] = TARGET_NSIDE
    primary.header["ORDERING"] = "RING"
    primary.header["COORDSYS"] = "G"
    primary.header["BUNIT"] = "MJy/sr"
    primary.header["VERSION"] = NUISANCE_VERSION
    primary.header["FREQ"] = NUISANCE_FREQ
    primary.header["NREAL"] = len(seed_pairs)
    primary.header["COMMENT"] = "Each image extension stores one IQU full-sky nuisance realization with shape (3, NPIX)."
    fits.HDUList([primary]).writeto(path, overwrite=False)

    for i, (noise_seed, cmb_res_seed) in enumerate(seed_pairs, start=1):
        print(f"[{i}/{len(seed_pairs)}] stitching noise_seed={noise_seed:04d} CMB_res_seed={cmb_res_seed:02d}")
        q, u = load_nuisance_patches_from_index(indices, path_by_key, noise_seed, cmb_res_seed)
        if trim > 0:
            q = q[:, trim:-trim, trim:-trim]
            u = u[:, trim:-trim, trim:-trim]
        nuisance_map, nuisance_mask = stitch_qu(q, u, pixfaces, weights)
        nuisance_map[0, nuisance_mask] = 0.0
        nuisance_map[:, ~nuisance_mask] = hp.UNSEEN

        header = fits.Header()
        header["EXTNAME"] = nuisance_extension_name(noise_seed, cmb_res_seed)
        header["NOISESEE"] = int(noise_seed)
        header["CMBSEE"] = int(cmb_res_seed)
        header["NSIDE"] = TARGET_NSIDE
        header["ORDERING"] = "RING"
        header["COORDSYS"] = "G"
        header["BUNIT"] = "MJy/sr"
        header["FIELDS"] = "I,Q,U"
        fits.append(path, nuisance_map.astype(np.float32), header=header)

    print("Saved all nuisance realizations:", path)


In [ ]:
# Observed map -> NSIDE=1024
observed_map = hp.read_map(str(OBSERVED_FITS_IN), field=(0, 1, 2), verbose=False)
observed_map = np.asarray(observed_map, dtype=float)

if hp.get_nside(observed_map[0]) != TARGET_NSIDE:
    observed_map = hp.ud_grade(observed_map, nside_out=TARGET_NSIDE)

observed_map[(observed_map == hp.UNSEEN) | ~np.isfinite(observed_map)] = hp.UNSEEN
save_iqu(OBSERVED_FITS, observed_map)

# Recovered Q/U patches -> NSIDE=1024
pixfaces_all = fits.open(FITS_PIXFACES)[0].data.astype(np.int64)
valid_indices = np.arange(pixfaces_all.shape[0])
indices, grouped = list_recovered_patch_files_by_index(RECOVERED_PATCHES_DIR, valid_indices)
patches = load_grouped_patches(indices, grouped)

if patches.shape[1] < 2:
    raise ValueError("Recovered patches must have channel 0=Q and channel 1=U")

trim = BORDERPIX - KEEPBORDER
pixfaces = pixfaces_all[indices]
if trim > 0:
    pixfaces = pixfaces[:, trim:-trim, trim:-trim]
    patches = patches[:, :, trim:-trim, trim:-trim]

weights = cosine_weights(patches.shape[2], patches.shape[3], KEEPBORDER)
recovered_map, recovered_mask = stitch_qu(patches[:, 0], patches[:, 1], pixfaces, weights)

# Recovered has no independent I here; copy observed I only for a valid IQU FITS container.
obs_mask = np.isfinite(observed_map[0]) & (observed_map[0] != hp.UNSEEN)
rec_mask = recovered_mask & obs_mask
recovered_map[0, rec_mask] = observed_map[0, rec_mask]
recovered_map[:, ~rec_mask] = hp.UNSEEN

save_iqu(RECOVERED_FITS, recovered_map)

In [ ]:
# All nuisance Q/U realizations -> one multi-extension FITS at NSIDE=1024
try:
    pixfaces_all
except NameError:
    pixfaces_all = fits.open(FITS_PIXFACES)[0].data.astype(np.int64)

write_all_nuisance_realizations(ALL_NUISANCE_FITS, pixfaces_all)
